In [ ]:
import pandas as pd
import numpy as np
import scipy
import warnings
import ray
import os
import json
import itertools
import glob
from tqdm import tqdm
from datetime import datetime, timedelta

import yaml

with open("config.yaml") as f:
    config = yaml.safe_load(f)

MIMIC_PATH = config["mimic_path"]
SEPTIC_SCHOCK_PATH = config["output_path"]
META_DATA_PATH = config["meta_data_path"]
ERROR_DATE = '4444-04-04 04:00:00'

In [ ]:
def get_relevant_intersection(truth_df, intersect_field, target_path, chunksize=10000):
    if chunksize > 0:
        filtered = pd.DataFrame()
        for chunk in pd.read_csv(target_path, chunksize=chunksize, dtype = str):
            filtered = pd.concat([filtered, chunk[chunk[intersect_field].isin(truth_df[intersect_field])]])
    else:
        ret = pd.read_csv(target_path, dtype = str)
        return ret[ret[intersect_field].isin(truth_df[intersect_field])]
    return filtered

In [ ]:
def find_time(hadm_id, chart_df, lab_df, time = None):
    def find_time_inner(bp_list, max_value):
        if time is None:
            finds = rel_measure[(rel_measure['itemid'].isin(bp_list)) & (rel_measure['valuenum'] < max_value)]
        else:
            finds = rel_measure[(rel_measure['itemid'].isin(bp_list)) & (rel_measure['valuenum'] < max_value) & (pd.to_datetime(rel_measure['charttime']) < time)]
        try:
            if time is None:
                return finds.iloc[1]['charttime']
            else:
                return finds.iloc[-1]['charttime']
        except IndexError:
            return ERROR_DATE
    sys_bp =  ['220050', '220179', '224167', '225309','227243']
    dial_bp = [ '220051', '220180', '224643', '225310', '227242']
    mean_bp = ['220052', '220181', '225312']
    all_bp = list()
    all_bp.extend(sys_bp)
    all_bp.extend(dial_bp)
    all_bp.extend(mean_bp)

    rel_charts = chart_df[chart_df['hadm_id'] == hadm_id]
    rel_charts = rel_charts[rel_charts['value'].notna()]
    rel_charts = rel_charts.sort_values(by='charttime')
    rel_measure = rel_charts[rel_charts['itemid'].isin(all_bp)]
    rel_measure['valuenum'] = rel_measure['valuenum'].astype('float')
    time_candidates = list()
    
    try:
        baseline = rel_measure[rel_measure['itemid'].isin(sys_bp)].iloc[0]['valuenum']
    except IndexError:
        baseline = 0
    time_candidates.append(find_time_inner(sys_bp, 90))
    time_candidates.append(find_time_inner(mean_bp, 65))
    time_candidates.append(find_time_inner(sys_bp, baseline - 40))
    
    if time is not None:
        try:
            time_candidates.append(lab_df[(lab_df['itemid'] == '50813') & (lab_df['hadm_id'] == hadm_id) & (lab_df.apply(lambda x: float(x['valuenum']) > 4, axis=1))].iloc[0]['charttime'])
        except IndexError:
            time_candidates.append(ERROR_DATE)
    return min(time_candidates)

In [ ]:
def dict_to_df(target):
    return pd.DataFrame(data=target)

In [ ]:
def create_checklist(hadm_id, hour_list, labevent_df, stay_df, proc_df, input_df, rel_items_df, chart_df):
    def find_info(df, subs_id, start_time, time_name, index = 0):
        try:
            if not subs_id.isnumeric():
                return df[(df['hadm_id'] == hadm_id) & (df['itemid'].isin(rel_items_df[rel_items_df['category'] == subs_id]['itemid'])) & (df.apply(lambda x: datetime.fromisoformat(x[time_name]) > start_time, axis=1))].iloc[index] # .sort_values(by=f'order_{time_name}')
            else:
                return df[(df['hadm_id'] == hadm_id) & (df['itemid'] == subs_id) & (df.apply(lambda x: datetime.fromisoformat(x[time_name]) > start_time, axis=1))].iloc[index] 
        except IndexError:
            return pd.DataFrame()
            
    def get_time(df, field_n):
        try:
            return  datetime.fromisoformat(df[field_n])
        except KeyError:
            return  datetime.fromisoformat(ERROR_DATE)
    lactate_id = '50813'
    blood_culture_id = '225401'
    time_zero =  datetime.fromisoformat(find_time(hadm_id, chart_df, lab_df))
    if time_zero >= datetime.fromisoformat(ERROR_DATE):
        print('Skip')
        return {}
    lactate_measure_info = find_info(lab_df, lactate_id, time_zero, 'charttime')
    blood_culture_info = find_info(proc_df, blood_culture_id, time_zero - timedelta(hours=12, minutes=0), 'starttime')
    antibiotic_inf = find_info(input_df, 'Antibiotics', time_zero - timedelta(hours=12, minutes=0), 'starttime')
    fluid_inf = find_info(input_df, 'Fluids', time_zero - timedelta(hours=1, minutes=0), 'starttime')
    vasopressor_inf = find_info(input_df, 'Medications', time_zero - timedelta(hours=1, minutes=0), 'starttime')
    
    measurement_time = get_time(lactate_measure_info, 'charttime')
    blood_culture_time = get_time(blood_culture_info, 'starttime')
    antibiotic_inf_time = get_time(antibiotic_inf, 'starttime')
    fluid_inf_time = get_time(fluid_inf, 'starttime')
    vasopressor_inf_time = get_time(vasopressor_inf, 'starttime')
    
    fluid_input = find_info(input_df, 'Fluids', time_zero - timedelta(hours=1, minutes=0), 'starttime', slice(0, len(input_df.index)))
    fluid_input['starttime'] = pd.to_datetime(fluid_input['starttime'])
    remeasure_time = (get_time(find_info(lab_df, lactate_id, time_zero, 'charttime', 1), 'charttime') - measurement_time)

    second_blodd = get_time(find_info(proc_df, blood_culture_id, time_zero - timedelta(hours=12, minutes=0), 'starttime', 1), 'starttime')
    
    results = dict()
    for hour in hour_list:
        fluid_max = time_zero + timedelta(hours=hour, minutes=0)
        
        fluid_input_loop = fluid_input[fluid_input['starttime'] < fluid_max]
        if not fluid_input_loop.empty:
            fluid_input_sum = fluid_input_loop['totalamount'].astype('float').sum()
            p_weight = float(fluid_input_loop['patientweight'].iloc[0])
            enough_fluid = fluid_input_sum >= p_weight * hour * 10
        else:
            enough_fluid = False

        if enough_fluid:
            fluid_cumsum = fluid_input_loop['totalamount'].astype('float').cumsum()
            finish_date = fluid_input_loop.loc[fluid_cumsum[fluid_cumsum >= p_weight * hour * 10].index[0]]['starttime']
            stayed_hypo = find_time(hadm_id, chart_df, lab_df, finish_date) != ERROR_DATE
        
        hour_in_delta = timedelta(hours=hour)
        vasp_init = vasopressor_inf_time - time_zero < hour_in_delta
        init_lactate = (measurement_time - time_zero < hour_in_delta and measurement_time > time_zero)
        rem_lactate = (remeasure_time <  timedelta(hours = 6)) or ('valuenum' in lactate_measure_info and float(lactate_measure_info['valuenum']) < 2)
        blood_cultures = (blood_culture_time - time_zero < hour_in_delta or (blood_culture_time < time_zero - timedelta(hours = 12))) and (second_blodd - time_zero < hour_in_delta or second_blodd < time_zero - timedelta(hours = 12))
        antibiotics = antibiotic_inf_time - time_zero < hour_in_delta or antibiotic_inf_time < time_zero - timedelta(hours = 12)
        blood_antibiotic_order = blood_culture_time < antibiotic_inf_time
        fluid_admin_start = fluid_inf_time - time_zero < hour_in_delta
        vasopressor_recom = vasp_init and enough_fluid and vasopressor_inf_time > finish_date and vasopressor_inf_time - finish_date < timedelta(minutes=30) and stayed_hypo
        results[hour] = dict_to_df({
            'subject_id' : [stay_df[stay_df['hadm_id'] == hadm_id]['subject_id'].iloc[0]],
            'hadm_id' : [hadm_id],
            'lactate_prod' : [init_lactate and rem_lactate],
            'blood_check_prod' : [blood_cultures and blood_antibiotic_order],
            'antibiotics_prod' : [antibiotics],
            'fluid_admin_prod' : [fluid_admin_start and enough_fluid],
            'vasopressor_init_prod' : [vasp_init], 
            'vasopressor_recom_prod' : [vasopressor_recom],
            'init_lactate_backup' : [init_lactate],
            'rem_lactate_backup' : [rem_lactate],
            'blood_cultures_backup' : [blood_cultures],
            'antibiotics_backup' : [antibiotics],
            'blood_antibiotic_order_backup' : [blood_antibiotic_order],
            'fluid_admin_start_backup' : [fluid_admin_start],
            'enough_fluid_backup' : [enough_fluid],
            'vasopressor_recom' : [vasopressor_recom],
        })
    return results

In [ ]:
with open(META_DATA_PATH, 'r', encoding='utf8') as fp:
    icd_codes =  pd.Series(itertools.chain(*json.load(fp).values()))

chunk = pd.read_csv(f'{MIMIC_PATH}/hosp/diagnoses_icd.csv', dtype = str)
chunk['icd_code'] = chunk['icd_code'].transform(lambda x:  (x[:3] + '.' + x[3:]).strip() if len(x.strip()) > 3 else x.strip())
filtered_df = pd.concat([chunk[chunk['icd_code'].isin(icd_codes)], chunk[chunk['icd_code'].str.startswith(tuple(icd_codes))]])
filtered_df = filtered_df.loc[filtered_df['icd_code'] != '008.45']
if not os.path.isdir(f'{SEPTIC_SCHOCK_PATH}'):
    os.mkdir(f'{SEPTIC_SCHOCK_PATH}')
filtered_df.to_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_diagnoses_icd.csv')
filtered_df

In [ ]:
get_relevant_intersection(diag_df, 'subject_id', f'{MIMIC_PATH}/hosp/patients.csv', -1).to_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_patients.csv')

In [ ]:
get_relevant_intersection(diag_df, 'hadm_id', f'{MIMIC_PATH}/icu/icustays.csv', -1).sort_values(by='intime').to_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_icu_stay.csv')

In [ ]:
stay_df = pd.read_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_icu_stay.csv', dtype=str)
stay_df

In [ ]:
grouped = (stay_df.groupby('subject_id').count() > 1).reset_index()
temp = stay_df[stay_df['subject_id'].isin(grouped[grouped['los'] == True]['subject_id'])]
to_drop = list()
for subject_id in temp['subject_id'].unique():
    to_drop.extend(temp[temp['subject_id'] == subject_id].iloc[1:].index.to_list())
stay_df = stay_df.drop(index=to_drop)
stay_df = stay_df[stay_df.apply(lambda x: (datetime.fromisoformat(x['outtime']) - datetime.fromisoformat(x['intime'])).days > 0, axis = 1)]
stay_df.to_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_icu_stay.csv')

In [ ]:
get_relevant_intersection(stay_df, 'subject_id', f'{MIMIC_PATH}/hosp/patients.csv', -1)

In [ ]:
get_relevant_intersection(stay_df, 'hadm_id', f'{MIMIC_PATH}/hosp/transfers.csv', -1).to_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_transfers.csv')

In [ ]:
fstay_df = pd.read_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_icu_stay.csv', dtype=str)

In [ ]:
get_relevant_intersection(fstay_df, 'hadm_id', f'{MIMIC_PATH}/icu/procedureevents.csv', -1).to_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_procedureevents.csv')

In [ ]:
get_relevant_intersection(fstay_df, 'hadm_id', f'{MIMIC_PATH}/icu/chartevents.csv', -1).to_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_chartevents.csv')

In [ ]:
lab_df = get_relevant_intersection(fstay_df, 'hadm_id', f'{MIMIC_PATH}/hosp/labevents.csv', -1).to_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_labevents.csv')

In [ ]:
get_relevant_intersection(fstay_df, 'hadm_id', f'{MIMIC_PATH}/icu/inputevents.csv', -1).to_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_inputs.csv')

In [ ]:
hour_one_df = pd.DataFrame()
hour_six_df = pd.DataFrame()

In [ ]:
for current_hadm_id in tqdm(fstay_df['hadm_id']):
    if current_hadm_id in done_ids:
        continue
    #current_res = create_checklist(current_hadm_id, [3,1], lab_df, fstay_df, proc_df, input_df, rel_items_df, chart_df)
    current_res = create_checklist(current_hadm_id, [6], lab_df, fstay_df, proc_df, input_df, rel_items_df, chart_df)
    if not current_res:
        continue
    #hour_one_df = pd.concat([hour_one_df, current_res[1]])
    #hour_three_df = pd.concat([hour_three_df, current_res[3]])
    hour_six_df = pd.concat([hour_three_df, current_res[6]])
#hour_one_df.to_csv(f'{SEPTIC_SCHOCK_PATH}/hour_one.csv')
#hour_three_df.to_csv(f'{SEPTIC_SCHOCK_PATH}/hour_three.csv')
hour_six_df.to_csv(f'{SEPTIC_SCHOCK_PATH}/hour_six.csv')